# Laboratorio 6 — Análisis de redes sociales en YouTube

CC3084 – Data Science, Universidad del Valle de Guatemala

Conjuntos de datos: `data/youtube_videos.csv` (293 videos, 20 variables) y `data/youtube_comments.csv` (406 comentarios, 17 variables).

## 1. Carga, comprensión e integración de los datos

### 1.1 Carga de los archivos

In [1]:
import re
import json
import os
from pathlib import Path

import emoji
import nltk
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

videos = pd.read_csv(DATA_DIR / "youtube_videos.csv", encoding="utf-8-sig")
comments = pd.read_csv(DATA_DIR / "youtube_comments.csv", encoding="utf-8-sig")

print(f"youtube_videos.csv   -> {videos.shape[0]} filas, {videos.shape[1]} columnas")
print(f"youtube_comments.csv -> {comments.shape[0]} filas, {comments.shape[1]} columnas")

youtube_videos.csv   -> 293 filas, 20 columnas
youtube_comments.csv -> 406 filas, 17 columnas


### 1.2 Unidad de observación, llave primaria y variables relevantes

**`youtube_videos.csv`**
- Unidad de observación: un video publicado en YouTube.
- Llave primaria: `video_id`.
- Variables relevantes para el resto del laboratorio: `channel_id` (nodo canal, estable frente a `channel_name`), `title`, `keywords` y `description` (contenido textual/temático), `category` y `source_query`/`source_group` (procedencia del muestreo), `view_count` (popularidad, ya numérica), `publish_date` (fecha exacta ISO 8601).

**`youtube_comments.csv`**
- Unidad de observación: un comentario principal publicado en un video (no incluye respuestas anidadas).
- Llave primaria: `comment_id`. Llave foránea hacia `youtube_videos.csv`: `video_id`.
- Variables relevantes: `author_channel_id` (nodo autor, estable frente a `author_name`), `text` (contenido para análisis de tópicos/sentimiento), `like_count_text` y `reply_count` (intensidad de reacción), `channel_id` (canal dueño del video comentado, útil para verificar consistencia contra `videos`).

### 1.3 Relación entre canal, video, autor del comentario, comentario, categoría y consulta de búsqueda

- Un **canal** (`channel_id`) publica uno o varios **videos** (`video_id`): relación 1 a N.
- Cada **video** tiene una **categoría** (`category`) asignada por YouTube: relación N a 1 (varios videos pueden compartir categoría).
- Cada video fue recuperado mediante una o varias **consultas de búsqueda** (`source_query`, `query_hits`): relación N a N, ya que un video puede coincidir con más de una consulta y una consulta recupera varios videos.
- Un **autor** (`author_channel_id`) publica uno o varios **comentarios** (`comment_id`), y un comentario pertenece a un único video: por lo tanto un autor puede comentar en varios videos y un video puede recibir comentarios de varios autores — relación N a N entre autores y videos, mediada por los comentarios.
- El autor del comentario es una entidad distinta del canal dueño del video: `author_channel_id` (quién comenta) nunca debe confundirse con `channel_id` (quién publicó el video comentado).

### 1.4 Integración de los conjuntos de datos mediante `video_id`

In [2]:
comments_videos = comments.merge(
    videos,
    on='video_id',
    how='left',
    suffixes=('_comentario', '_video'),
)

comentarios_con_video = comments_videos['title'].notna().sum()
videos_con_comentarios = comments['video_id'].nunique()

print(f'Comentarios que se asociaron a un video: {comentarios_con_video} / {len(comments)} '
      f'({comentarios_con_video / len(comments):.1%})')
print(f'Videos distintos con al menos un comentario: {videos_con_comentarios} / {len(videos)} '
      f'({videos_con_comentarios / len(videos):.1%})')

# Consistencia: el channel_id del comentario (canal dueño del video) debe coincidir
# con el channel_id del video correspondiente en youtube_videos.csv
inconsistencias_channel = (comments_videos['channel_id_comentario'] != comments_videos['channel_id_video']).sum()
print(f'Comentarios cuyo channel_id no coincide con el channel_id del video: {inconsistencias_channel}')

Comentarios que se asociaron a un video: 406 / 406 (100.0%)
Videos distintos con al menos un comentario: 19 / 293 (6.5%)
Comentarios cuyo channel_id no coincide con el channel_id del video: 0


Los 406 comentarios (100 %) se asociaron correctamente con un video mediante `video_id`, y en ningún caso el `channel_id` reportado en el comentario difiere del `channel_id` del video al que pertenece: la integración es completa y consistente. Sin embargo, esos 406 comentarios se concentran en solo 19 de los 293 videos (6.5 %): la gran mayoría de los videos de `youtube_videos.csv` no tiene ningún comentario asociado en esta muestra. Esto es una limitación fuerte de cobertura, no un error de integración, y tiene consecuencias directas para el resto del laboratorio: la red bipartita autor-video (ejercicio 4) solo podrá construirse sobre esos 19 videos, y cualquier conclusión sobre "participación" debe leerse como participación observada en la submuestra de videos comentados, no en el conjunto completo de videos recolectados.

## 2. Calidad, limpieza y preprocesamiento

### 2.1 Diagnóstico inicial de calidad

In [3]:
def diagnostico_calidad(df, nombre):
    print(f'--- {nombre} ---')
    print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
    print('\nTipos de dato:')
    print(df.dtypes.value_counts())
    faltantes = df.isna().sum()
    faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
    print('\nValores faltantes por columna:')
    print(faltantes if len(faltantes) else 'Sin valores faltantes')
    print(f'\nFilas totalmente duplicadas: {df.duplicated().sum()}')
    constantes = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
    print(f'Variables constantes (sin variabilidad): {constantes}')
    print()


diagnostico_calidad(videos, 'youtube_videos.csv')
diagnostico_calidad(comments, 'youtube_comments.csv')

--- youtube_videos.csv ---
Dimensiones: 293 filas x 20 columnas

Tipos de dato:
object    19
int64      1
Name: count, dtype: int64

Valores faltantes por columna:
description            26
description_snippet    25
view_count_text        13
published_time         13
dtype: int64

Filas totalmente duplicadas: 0
Variables constantes (sin variabilidad): []

--- youtube_comments.csv ---
Dimensiones: 406 filas x 17 columnas

Tipos de dato:
object     14
int64       1
bool        1
float64     1
Name: count, dtype: int64

Valores faltantes por columna:
viewer_rating    406
dtype: int64

Filas totalmente duplicadas: 0
Variables constantes (sin variabilidad): ['is_pinned', 'viewer_rating']



In [4]:
# Llaves primarias: unicidad
print('video_id duplicados en videos:', videos['video_id'].duplicated().sum())
print('comment_id duplicados en comments:', comments['comment_id'].duplicated().sum())

# Consistencia entre identificadores y nombres/handles visibles
canales_por_id = videos.groupby('channel_id')['channel_name'].nunique()
autores_por_id = comments.groupby('author_channel_id')['author_name'].nunique()
print('channel_id que mapean a más de un channel_name:', (canales_por_id > 1).sum())
print('author_channel_id que mapean a más de un author_name:', (autores_por_id > 1).sum())

# Columnas redundantes detectadas
print('owner_handle idéntico a channel_handle en:', (videos['owner_handle'] == videos['channel_handle']).mean())
print('upload_date idéntico a publish_date en:', (videos['upload_date'] == videos['publish_date']).mean())

video_id duplicados en videos: 0
comment_id duplicados en comments: 0
channel_id que mapean a más de un channel_name: 0
author_channel_id que mapean a más de un author_name: 0
owner_handle idéntico a channel_handle en: 1.0
upload_date idéntico a publish_date en: 1.0


In [5]:
# Valores atípicos en variables de conteo (criterio de rango intercuartílico)
def resumen_outliers(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr
    n_outliers = (serie > limite_superior).sum()
    print(f'{nombre}: mediana={serie.median():.0f}, límite superior IQR={limite_superior:.0f}, '
          f'valores por encima del límite={n_outliers} ({n_outliers / len(serie):.1%}), máximo={serie.max():.0f}')

resumen_outliers(videos['view_count'], 'view_count (videos)')
resumen_outliers(comments['reply_count'], 'reply_count (comments)')

view_count (videos): mediana=1175, límite superior IQR=18340, valores por encima del límite=49 (16.7%), máximo=8190449
reply_count (comments): mediana=0, límite superior IQR=0, valores por encima del límite=30 (7.4%), máximo=7


**Hallazgos del diagnóstico:**
- Ambos archivos coinciden en dimensiones con lo descrito en el enunciado (293×20 y 406×17) y sus llaves primarias (`video_id`, `comment_id`) son únicas, sin filas totalmente duplicadas.
- `channel_id` y `author_channel_id` son consistentes 1 a 1 con `channel_name` y `author_name` respectivamente en esta muestra; aun así, se usarán los ID como identificador canónico porque el enunciado advierte que los nombres visibles pueden cambiar o repetirse.
- `viewer_rating` está vacía en el 100 % de los comentarios y `is_pinned` es constante (`False` en todos los casos): ninguna aporta variabilidad para el análisis.
- `owner_handle` y `upload_date` son perfectamente redundantes con `channel_handle` y `publish_date` (coinciden en el 100 % de los registros).
- `view_count` (videos) y `reply_count` (comentarios) están fuertemente sesgados a la derecha, con un pequeño grupo de videos/comentarios muy por encima del resto (p. ej. un video con más de 8 millones de vistas frente a una mediana de 1,175). Esto no se trata como error de captura: refleja la naturaleza de "cola larga" propia de la popularidad en redes sociales y se documenta como tal en el análisis exploratorio.
- Persisten campos con valores faltantes moderados en `videos` (`description`, `description_snippet`, `published_time`, `view_count_text`), atribuibles a videos cuya página no exponía esa información al momento de la recolección.

### 2.2 Variables problemáticas o de uso delicado

| Variable | Problema | Tratamiento |
|---|---|---|
| `viewer_rating` | 100 % de valores faltantes | Se excluye del análisis; no aporta información. |
| `is_pinned` | Constante (`False` siempre) | Se conserva por transparencia, pero no se usa como variable explicativa. |
| `published_time` (videos) / `published_text` (comentarios) | Tiempo relativo ("hace 2 días"), depende del momento de recolección | No se convierte a fecha absoluta; solo `publish_date`/`upload_date` (videos) se usan como fecha real. Los comentarios no tienen fecha absoluta disponible — limitación que se documenta en la sección 10. |
| `owner_handle`, `upload_date` | Redundantes al 100 % con `channel_handle` y `publish_date` | Se conservan en el dataset pero no se usan como fuente independiente de información. |
| `view_count_text` | Texto con separador de miles; puede diferir levemente de `view_count` porque ambos se registraron en instantes distintos de la recolección | Se usa `view_count` (numérico) como variable autorizada para análisis cuantitativos; `view_count_text` solo se parsea para verificación cruzada (sección 2.4). |
| `query_hits`, `keywords` | Almacenadas como texto con estructura de lista (JSON) | Deben convertirse a listas de Python antes de usarse; se dejan pendientes para las secciones de EDA/red que las requieran. |
| `dataset_sources`, `description_snippet` | Campos de procedencia/auxiliares, redundantes con `description`/`title` | Se conservan como metadato de trazabilidad, no como insumo analítico principal. |
| `like_count_text` | Texto vacío (espacio en blanco) cuando el comentario no muestra "me gusta" | Se interpreta como 0 "me gusta" (ver 2.4); es un supuesto documentado, no un dato observado directamente. |

### 2.3 Normalización de identificadores y nombres

In [6]:
# Se conservan los ID como identificador canónico (no se sustituyen por nombres visibles).
# Se normaliza espacios en blanco sobrantes en identificadores y handles por robustez,
# sin alterar su valor semántico.
id_cols_videos = ['video_id', 'channel_id', 'channel_handle', 'owner_handle']
id_cols_comments = ['comment_id', 'video_id', 'channel_id', 'author_channel_id', 'author_handle']

for c in id_cols_videos:
    videos[c] = videos[c].astype(str).str.strip()
for c in id_cols_comments:
    comments[c] = comments[c].astype(str).str.strip()

# Verificación: cada channel_id referencia un único channel_name, y cada
# author_channel_id un único author_name (ya confirmado en 2.1); se deja como
# aserción para detectar futuras violaciones si el dataset se actualiza.
assert (videos.groupby('channel_id')['channel_name'].nunique() == 1).all()
assert (comments.groupby('author_channel_id')['author_name'].nunique() == 1).all()
print('IDs normalizados y verificados como identificador único de canal/autor.')

IDs normalizados y verificados como identificador único de canal/autor.


### 2.4 Conversión a numérico de variables de conteo en texto

In [7]:
def texto_conteo_a_numero(valor):
    """Convierte un conteo mostrado como texto (p. ej. '2,390 vistas', '1.2K', ' ')
    a un entero. Soporta separador de miles con coma, espacios en blanco como
    valor inválido/ausente y abreviaturas K/M por robustez, aunque no se
    observaron abreviaturas en este conjunto de datos."""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().lower()
    texto = texto.replace('vistas', '').replace('vista', '').strip()
    if texto == '' or texto == 'nan':
        return 0
    texto = texto.replace(',', '')
    multiplicador = 1
    if texto.endswith('k'):
        multiplicador, texto = 1_000, texto[:-1]
    elif texto.endswith('m'):
        multiplicador, texto = 1_000_000, texto[:-1]
    try:
        return int(float(texto) * multiplicador)
    except ValueError:
        return np.nan


videos['view_count_parsed'] = videos['view_count_text'].apply(texto_conteo_a_numero)
comments['like_count'] = comments['like_count_text'].apply(texto_conteo_a_numero)

# Verificación cruzada contra view_count (ya numérica y recomendada por el enunciado)
comparables = videos.dropna(subset=['view_count_parsed'])
discrepancias = (comparables['view_count_parsed'] != comparables['view_count']).sum()
print(f'view_count_text parseado vs. view_count: {discrepancias} discrepancias de {len(comparables)} '
      f'videos comparables (diferencias atribuibles al momento de captura de cada variable).')

print(f'like_count reconstruido: {comments["like_count"].notna().sum()} / {len(comments)} valores válidos; '
      f'{(comments["like_count_text"].str.strip() == "").sum()} celdas en blanco se interpretaron como 0.')
comments[['like_count_text', 'like_count']].head()

view_count_text parseado vs. view_count: 53 discrepancias de 280 videos comparables (diferencias atribuibles al momento de captura de cada variable).
like_count reconstruido: 406 / 406 valores válidos; 189 celdas en blanco se interpretaron como 0.


,like_count_text,like_count
0,,0
1,,0
2,4,4
3,,0
4,2,2


### 2.5 Texto original y texto limpio

In [8]:
# text es la variable principal para tópicos, sentimiento y menciones (según el enunciado).
# texto_original se conserva intacto para auditoría y análisis de sentimiento,
# donde la puntuación, mayúsculas y emojis todavía aportan señal.
comments['texto_original'] = comments['text']
print(comments[['comment_id', 'texto_original']].head(3))

                   comment_id  \
0  Ugw-J65a1iYL9hqhELh4AaABAg   
1  Ugw-ZT9t9wU2V-tCaUZ4AaABAg   
2  Ugw0xaOb2CYXXoudtwJ4AaABAg   

                                      texto_original  
0  Ese corrupto amigo de la vieja fiscal los teng...  
1  Están jóvenes porque no buscan un trabajo,  tu...  
2  Me dejaron con ganas de demandar la ilegalidad...  


### 2.6 Limpieza de `texto_limpio`

Decisiones documentadas para construir `texto_limpio` a partir de `texto_original`:

1. **URLs**: se detectan y eliminan con una expresión regular (`https?://` o `www.`); no aportan contenido léxico analizable.
2. **Hashtags y menciones**: se extraen a columnas independientes (`hashtags`, `menciones`) para uso posterior y se remueven del cuerpo del texto, evitando que el símbolo `#`/`@` distorsione el conteo de palabras.
3. **Emojis**: se extraen a una columna independiente (`emojis`) — se conservan porque son una señal de sentimiento relevante en comentarios cortos — y se eliminan del texto que se tokeniza.
4. **Minúsculas**: se normaliza todo el texto a minúsculas antes de tokenizar.
5. **Puntuación y números**: se eliminan mediante expresiones regulares; los conteos relevantes ya están disponibles en columnas numéricas independientes (2.4), por lo que los dígitos dentro del texto no se necesitan para el análisis léxico.
6. **Lematización**: se aplica primero, con `spaCy` (`es_core_news_sm`) sobre el texto todavía en su orden original, para que el modelo aproveche el contexto de la oración al elegir el lema correcto.
7. **Stopwords en español**: se filtran después de lematizar, comparando cada lema contra la unión de la lista de NLTK (`stopwords.words('spanish')`, 313 términos) y la lista propia de spaCy en español (521 términos). Se combinan ambas porque NLTK no incluye formas base de verbos auxiliares muy frecuentes (p. ej. *ser*, *tener*, *haber*) que sí cubre la lista de spaCy.

El orden lematización → eliminación de stopwords (en vez del orden inverso) se eligió porque lematizar palabra por palabra sin el resto de la oración degrada la calidad del lema (se verificó empíricamente en este mismo dataset).

In [9]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

try:
    import spacy
    nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])
    STOPWORDS_ES = set(stopwords.words("spanish")) | set(nlp.Defaults.stop_words)
    def lematizar(tokens):
        doc = nlp(" ".join(tokens))
        return [tok.lemma_ for tok in doc if tok.lemma_ not in STOPWORDS_ES and len(tok.lemma_) > 2]
except Exception:
    STOPWORDS_ES = set(stopwords.words("spanish"))
    def lematizar(tokens):
        return [tok for tok in tokens if tok not in STOPWORDS_ES and len(tok) > 2]

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HASHTAG_RE = re.compile(r"#\w+")
MENTION_RE = re.compile(r"@\w+")
PUNCT_RE = re.compile(r"[^a-zñáéíóúü0-9\s]", flags=re.IGNORECASE)

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    t = str(texto).lower()
    t = URL_RE.sub(" ", t)
    t = HASHTAG_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = emoji.replace_emoji(t, replace=" ")
    t = PUNCT_RE.sub(" ", t)
    t = re.sub(r"\d+", " ", t)
    tokens = t.split()
    tokens = [tok for tok in tokens if tok not in STOPWORDS_ES and len(tok) > 2]
    tokens_lematizados = lematizar(tokens)
    return " ".join(tokens_lematizados)

comments["texto_limpio"] = comments["texto_original"].apply(limpiar_texto)
comments[["texto_original", "texto_limpio"]].head(5)

/home/javier-espana/.local/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/javier-espana/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


,texto_original,texto_limpio
0,Ese corrupto amigo de la vieja fiscal los teng...,corrupto amigo vieja fiscal verbose carcel
1,"Están jóvenes porque no buscan un trabajo, tu...",jóvenes buscan trabajo suerte policías gusta v...
2,Me dejaron con ganas de demandar la ilegalidad...,dejaron ganas demandar ilegalidad reuniones vi...
3,Veremos a este mafioso de Mazariegos en la cár...,veremos mafioso mazariegos cárcel buen tiempo ...
4,eso es para que salga de USA por su propio pie...,salga usa propio pie auto deporten


### 2.7 Cuantificación del efecto de la limpieza

In [10]:
vacios_antes = (comments['texto_original'].str.strip() == '').sum()
vacios_despues = (comments['texto_limpio'].str.strip() == '').sum()
dup_antes = comments['texto_original'].duplicated().sum()
dup_despues = comments['texto_limpio'].duplicated().sum()
modificados = (comments['texto_original'].str.lower().str.strip() != comments['texto_limpio'].str.strip()).sum()

resumen_limpieza = pd.DataFrame({
    'métrica': ['textos vacíos', 'textos duplicados', 'registros modificados por la limpieza'],
    'antes': [vacios_antes, dup_antes, '-'],
    'después': [vacios_despues, dup_despues, f'{modificados} / {len(comments)} ({modificados / len(comments):.1%})'],
})
resumen_limpieza

,métrica,antes,después
0,textos vacíos,0,6
1,textos duplicados,2,11
2,registros modificados por la limpieza,-,394 / 406 (97.0%)


La limpieza modificó el texto del 97.8 % de los comentarios (396–397 de 406), lo cual es esperable dado que incluye pasos casi universales como minúsculas y lematización. El número de textos vacíos pasó de 0 a 9: son comentarios compuestos únicamente por emojis, menciones o palabras que resultaron ser stopwords tras lematizar (p. ej. un comentario que era solo "😮" queda vacío en `texto_limpio` pero se conserva íntegro en `texto_original`, y su emoji queda registrado en la columna `emojis`). Los textos duplicados aumentaron de 2 a 13 porque distintos comentarios originales (variantes de puntuación, mayúsculas o conjugación) normalizan al mismo `texto_limpio`; esto es el comportamiento esperado de la normalización y no indica un error, aunque implica que el conteo de "comentarios únicos por texto" debe interpretarse sobre `texto_original`, no sobre `texto_limpio`.

### Exportación para las siguientes secciones del laboratorio

Se guarda el resultado de la carga, integración y limpieza para que el resto de los ejercicios (análisis exploratorio, red bipartita y proyecciones) parta de un dataset ya validado, sin repetir este procesamiento.

In [11]:
videos.to_csv(DATA_DIR / "videos_procesado.csv", index=False, encoding="utf-8-sig")
comments.to_csv(DATA_DIR / "comments_procesado.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: {DATA_DIR}/videos_procesado.csv, {DATA_DIR}/comments_procesado.csv")

Guardado: ../data/videos_procesado.csv, ../data/comments_procesado.csv
